In [ ]:
import fitz  # PyMuPDF
import json
import uuid

def extract_bookmarks_json(pdf_path):
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()  # Get bookmarks (TOC)

    if not toc:
        print("No bookmarks found in the PDF.")
        return

    chapters = []
    chapter_stack = {}

    for entry in toc:
        level, title, start_page = entry
        chapter_id = str(uuid.uuid4())  # Generate unique ID for each chapter

        if level == 1:
            # Create a new chapter
            chapter_stack[level] = {
                "chapter": {
                    "title": title,
                    "start_page": start_page,
                    "end_page": None,  # Will be set later
                    "subchapters": []
                }
            }
            chapters.append(chapter_stack[level])

        elif level == 2:
            # Add a subchapter to the last main chapter
            if 1 in chapter_stack:
                chapter_stack[level] = {
                    "title": title,
                    "start_page": start_page,
                    "end_page": None  # Will be set later
                }
                chapter_stack[1]["chapter"]["subchapters"].append(chapter_stack[level])

    # Assign end_page based on next entry
    for i in range(len(chapters)):
        if i < len(chapters) - 1:
            chapters[i]["chapter"]["end_page"] = chapters[i + 1]["chapter"]["start_page"] - 1
        else:
            chapters[i]["chapter"]["end_page"] = doc.page_count  # Last chapter ends at last page

        for j in range(len(chapters[i]["chapter"]["subchapters"])):
            if j < len(chapters[i]["chapter"]["subchapters"]) - 1:
                chapters[i]["chapter"]["subchapters"][j]["end_page"] = chapters[i]["chapter"]["subchapters"][j + 1]["start_page"] - 1
            else:
                chapters[i]["chapter"]["subchapters"][j]["end_page"] = chapters[i]["chapter"]["end_page"]

    # Convert to JSON format
    output_json = json.dumps(chapters, indent=4)
    print(output_json)

# Example usage
pdf_file = "Sensors for Chemical and Biological Applications (Manoj Kumar Ram, Venkat R. Bhethanabotla).pdf"  # Replace with your PDF file path
extract_bookmarks_json(pdf_file)


[
    {
        "chapter": {
            "title": "0849333660",
            "start_page": 1,
            "end_page": 1,
            "subchapters": []
        }
    },
    {
        "chapter": {
            "title": "Sensors for\rChemical\rand Biological\rApplications",
            "start_page": 2,
            "end_page": 11,
            "subchapters": [
                {
                    "title": "Contents",
                    "start_page": 5,
                    "end_page": 6
                },
                {
                    "title": "Preface",
                    "start_page": 7,
                    "end_page": 8
                },
                {
                    "title": "List of Contributors",
                    "start_page": 9,
                    "end_page": 11
                }
            ]
        }
    },
    {
        "chapter": {
            "title": "Chapter 1: Solid-State Gas Sensors",
            "start_page": 12,
            "end_page": 53,
           